In [1]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
import torchaudio
import os
from nanodrz.utils import play
import torch

path = "/home/harry/.cache/nanodrz/small/1905/shortpoetry_052_librivox_64kb_mp3/onafavoritecat_gray_jb_64kb.flac"

In [13]:
audio,sr  = torchaudio.load(path)

In [14]:
silence_threshold=0.01
min_silence_len=0.2
min_chunk_len=1

In [31]:
audio = torch.cat([audio, torch.zeros(1, int(sr * min_silence_len) + 1)], dim=-1)
amplitude = torch.abs(audio)
is_silence = amplitude < silence_threshold
silent_frames = is_silence.all(dim=0)

import torch.nn.functional as F

# Define the kernel for convolution
kernel_size = int(0.2 * sr)
kernel = torch.ones(1, 1, kernel_size)

# Perform convolution to find silence frames
conv_output = F.conv1d(audio[None], kernel)


tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [34]:

idxs = (conv_output == 1).nonzero()[:,2]
idxs

tensor([ 372526, 1037154, 1045099, 1092687, 1153727, 1401642, 1702034, 2199360,
        2397164, 2472809, 2735630])

In [36]:
play(audio[:, int(idxs[0]-kernel_size/2):int(idxs[0]+kernel_size/2)])

SyntaxError: closing parenthesis ']' does not match opening parenthesis '(' (2381583822.py, line 1)

In [20]:
silent_frames

tensor([], dtype=torch.int64)

In [4]:
def find_nonsilence_chunks(
    audio_file: str,
    silence_threshold=0.01,
    min_silence_len=0.2,
    min_chunk_len=1,
):
    """
    Finds and returns non-silence chunks in the given audio.

    Args:
        audio (Tensor): The audio waveform.
        sr (int): The sample rate of the audio.
        silence_threshold (float, optional): The threshold below which audio is considered as silence. Defaults to 0.01.
        min_silence_len (float, optioanal): The minimum duration of silence to be considered as a separate chunk. Defaults to 0.2.
        min_chunk_len (float, optional): The minimum duration of a non-silence chunk. Defaults to 1.

    Returns:
        List[Tensor]: A list of non-silence chunks.
        List[Tuple[int, int]]: A list of tuples representing the start and end indexes of silence segments.
    """

    audio, sr = torchaudio.load(audio_file)
    # Add min_silence_len+1 silence to the end of the audio
    audio = torch.cat([audio, torch.zeros(1, int(sr * min_silence_len) + 1)], dim=-1)
    amplitude = torch.abs(audio)
    is_silence = amplitude < silence_threshold
    silent_frames = is_silence.all(dim=0)

    silence_indexes = []
    start_idx = 0

    for idx, is_silent in enumerate(silent_frames):
        if is_silent and start_idx == -1:
            start_idx = idx
        elif not is_silent and start_idx != -1:
            if (idx - start_idx) / sr >= min_silence_len:
                silence_indexes.append((start_idx, idx))
            start_idx = -1

    if (
        start_idx is not None
        and (len(silent_frames) - start_idx) / sr >= min_silence_len
    ):
        silence_indexes.append((start_idx, len(silent_frames)))

    chunks = []
    cur_chunk = torch.zeros(1, 0)
    cur_idx = 0

    for b, e in silence_indexes:
        cur_chunk = torch.cat([cur_chunk, audio[:, cur_idx:b]], dim=-1)
        if cur_chunk.shape[-1] > sr * min_chunk_len:
            cur_idx = e
            chunks.append(cur_chunk)
            cur_chunk = torch.zeros(1, 0)
        else:
            cur_idx = b

    if cur_chunk.shape[-1] != 0:
        chunks.append(cur_chunk)

    chunk_files = []
    bn = os.path.basename(audio_file)
    ext = bn.split(".")[-1]
    bn = bn.replace(ext, "")
    os.makedirs(os.path.join(CACHE_DIR, "chunks"), exist_ok=True)

    for i, c in enumerate(chunks):
        f = bn + "_" + str(i)
        p = os.path.join(CACHE_DIR, "chunks", f"{bn}_{str(i)}.{ext}")
        torchaudio.save(p, c, sr)
        chunk_files.append(f)

    return chunk_files